# Fine-grained comparison — Qwen3.5-0.8B vs LFM2.5-VL-1.6B, per scenario

For **every probe sample** this shows, side by side: the **image**, the **question**, the **ground
truth** (with its box / rationale), and **each model's prediction + score**. The point is to make it
easy to spot **flaws in the ground truth or the metric** — not just model errors.

Heuristic flag: when **both models agree with each other but disagree with the gold**, the GT/metric
is the likely culprit, not the models. Those are surfaced in the *GT-flaw suspects* table at the end.

Pick a benchmark with `BENCH`. Training-free: it runs the two base models as-is (GPU), or loads
cached `predictions.jsonl` if present (`LIVE=False`).

In [ ]:
# --- install docvlm_eval (fresh env e.g. Colab: clone+checkout; then editable install) ---
import os, sys, subprocess, importlib
from pathlib import Path

def _repo_root():
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "pyproject.toml").exists():
            return c
    return None

root = _repo_root()
if root is None:                       # fresh environment (Colab/Kaggle): clone the repo
    subprocess.run(["git", "clone", "https://github.com/SangbumChoi/OCR.git"], check=False)
    root = Path("OCR")
    subprocess.run(["git", "-C", str(root), "checkout", "claude/new-session-w79q0i"], check=False)
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
os.chdir(root)                         # cwd is now the repo root
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[models]"], check=True)

# The editable-install .pth is only read at interpreter startup, so a running kernel can't import
# the package until we add src/ to sys.path ourselves (avoids "No module named docvlm_eval").
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
import docvlm_eval
print("docvlm_eval ready from", docvlm_eval.__file__)


## 0. Setup + load the chosen probe

In [ ]:
import sys, json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

ROOT = Path.cwd()
if not (ROOT/"scripts").exists() and (ROOT.parent/"scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/"src"))
from docvlm_eval.benchmarks import load_jsonl
from docvlm_eval.metrics.text import score_sample
from docvlm_eval.metrics.grounding import parse_gold_box, parse_pred_box

MODELS = {"qwen3_5-0.8b": "Qwen/Qwen3.5-0.8B", "lfm2_5-vl-1.6b": "LiquidAI/LFM2.5-VL-1.6B"}
BENCHES = {
  "capability": "data/probes/capability_probe/capability.jsonl",
  "spatial":    "data/probes/spatial_context_probe/probe.jsonl",
  "realistic":  "data/probes/realistic_cases/realistic_cases.jsonl",
  "custom_eval":"data/probes/custom_eval/custom_eval.jsonl",
}
BENCH = "capability"     # <- change me
LIVE  = True             # True: run models on GPU; False: load cached predictions.jsonl
N     = 12               # max scenarios to render (None = all)

samples = load_jsonl(str(ROOT/BENCHES[BENCH]))
if N: samples = samples[:N]
print(f"{BENCH}: {len(samples)} scenarios")

## 1. Get predictions from both models (sequential = memory-safe on a T4)

In [ ]:
def _predict_live(hf_id, samples):
    import torch
    from transformers import AutoProcessor
    try:
        from transformers import AutoModelForImageTextToText as A
    except ImportError:
        from transformers import AutoModelForVision2Seq as A
    proc = AutoProcessor.from_pretrained(hf_id, trust_remote_code=True)
    model = A.from_pretrained(hf_id, torch_dtype=torch.bfloat16, trust_remote_code=True).cuda().eval()
    out = {}
    for s in samples:
        img = Image.open(s.image_path).convert("RGB")
        msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":s.question}]}]
        inp = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                       return_dict=True, return_tensors="pt").to("cuda")
        n = inp["input_ids"].shape[1]
        with torch.no_grad():
            g = model.generate(**inp, max_new_tokens=64, do_sample=False)
        out[s.sample_id] = proc.batch_decode(g[:, n:], skip_special_tokens=True)[0].strip()
    del model
    torch.cuda.empty_cache()
    return out

def _predict_cached(key, samples):
    f = ROOT/"docs"/"results"/key/BENCH/"predictions.jsonl"
    if not f.exists(): f = ROOT/"docs"/"results"/key/("probe" if BENCH=="spatial" else BENCH)/"predictions.jsonl"
    if not f.exists(): return {s.sample_id: "" for s in samples}
    d = {json.loads(l)["sample_id"]: json.loads(l)["prediction"] for l in f.read_text().splitlines() if l.strip()}
    return {s.sample_id: d.get(s.sample_id, "") for s in samples}

PRED = {}
for key, hf in MODELS.items():
    print("->", key, "(live)" if LIVE else "(cached)")
    PRED[key] = _predict_live(hf, samples) if LIVE else _predict_cached(key, samples)

# build per-scenario records with scores
KEYS = list(MODELS)
REC = []
for s in samples:
    r = {"id": s.sample_id, "q": s.question, "gold": s.answers, "metric": s.metric,
         "axis": s.answer_type, "image": s.image_path, "meta": s.meta}
    for k in KEYS:
        p = PRED[k][s.sample_id]
        r[k] = p
        r[k+"_score"] = round(score_sample(s.metric, p, s.answers), 3)
    REC.append(r)
print("done; scored", len(REC), "scenarios")

## 2. Per-scenario cards — image · question · GT · both predictions · score

In [ ]:
def _short(t, n=140):
    t = (t or "").replace("\n", " ")
    return t if len(t) <= n else t[:n] + "…"

def show_scenario(r):
    img = Image.open(r["image"]).convert("RGB")
    fig, ax = plt.subplots(figsize=(6, 6*img.height/img.width)); ax.imshow(img); ax.set_axis_off()
    # grounding overlay: GT (green) vs each model's parsed box (qwen=orange, lfm=blue)
    if r["metric"] == "grounding":
        gb = parse_gold_box(r["gold"][0])
        if gb:
            (gx, sz) = gb
            ax.add_patch(mpatches.Rectangle((gx[0],gx[1]), gx[2]-gx[0], gx[3]-gx[1],
                         fill=False, edgecolor="#1a9641", lw=2.2, label="GT"))
            for k, col in [(KEYS[0], "#d7791d"), (KEYS[1], "#2c7bb6")]:
                pb = parse_pred_box(r[k], sz)
                if pb: ax.add_patch(mpatches.Rectangle((pb[0],pb[1]), pb[2]-pb[0], pb[3]-pb[1],
                                    fill=False, edgecolor=col, lw=1.6, linestyle="--"))
        ax.legend(fontsize=7, loc="upper right")
    ax.set_title(f"{r['id']}  ·  axis={r['axis']}  ·  metric={r['metric']}", fontsize=10)
    plt.tight_layout(); plt.show()

    print(f"Q   : {_short(r['q'])}")
    print(f"GT  : {r['gold']}")
    if r["meta"].get("rationale"): print(f"why : {_short(r['meta']['rationale'])}")
    for k in KEYS:
        mark = "✅" if r[k+"_score"] >= 0.5 else "❌"
        print(f"{mark} {k:16}: {_short(r[k])}   (score={r[k+'_score']})")
    # quick flaw hint inline
    if r[KEYS[0]+"_score"] < 0.5 and r[KEYS[1]+"_score"] < 0.5:
        agree = score_sample("anls", r[KEYS[0]], [r[KEYS[1]]])
        if agree >= 0.6 and r[KEYS[0]].strip():
            print("⚠  both models agree with EACH OTHER but not the GT → check the ground truth/metric")
    print("-"*90)

for r in REC:
    show_scenario(r)

## 3. GT-flaw suspects (ranked)

Rules: **(A)** both models score &lt;0.5 *and* agree with each other (ANLS ≥ 0.6) → the gold/metric
is the likely problem; **(B)** the gold string is contained in a prediction yet scored 0 → a
metric/normalisation artefact (not a wrong answer). Eyeball these first.

In [ ]:
def agreement(a, b): return score_sample("anls", a or "", [b or ""])

suspects = []
for r in REC:
    sA, sB = r[KEYS[0]+"_score"], r[KEYS[1]+"_score"]
    pA, pB = r[KEYS[0]], r[KEYS[1]]
    gold = " | ".join(r["gold"])
    if sA < 0.5 and sB < 0.5 and pA.strip() and agreement(pA, pB) >= 0.6:
        suspects.append((agreement(pA, pB), "A: models agree, GT differs", r))
    elif any(g.strip().lower() in (pA+pB).lower() for g in r["gold"] if g.strip()) and max(sA, sB) < 0.5:
        suspects.append((0.5, "B: gold ⊆ prediction but scored 0 (metric)", r))

suspects.sort(key=lambda x: -x[0])
print(f"{len(suspects)} suspect scenario(s):\n")
for conf, why, r in suspects:
    print(f"[{why}]  {r['id']}  (axis={r['axis']}, metric={r['metric']})")
    print(f"   Q   : {_short(r['q'], 90)}")
    print(f"   GT  : {r['gold']}")
    print(f"   {KEYS[0]}: {_short(r[KEYS[0]], 70)}  | {KEYS[1]}: {_short(r[KEYS[1]], 70)}")
    print()
if not suspects:
    print("No obvious GT/metric flaws by these rules — but still skim the cards above.")